# NB-07 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-07**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-07.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This applies the observability discussion to a specific prediction target, comparing retained sensor designs and model-mismatch results.

- [Volume III, Chapter 7 — Sampling, PMU placement, and reconstruction](https://sherandoresearch.github.io/power-markets-series/vol3_spectral_grids.html#c7)
- [Volume IV, Chapter 13 — Sampling: three sensors, twenty-six buses](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#chiv-13)
- [Volume VI, Chapter 5 — Information and identifiability](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c5)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-07', require_key=True)
print(setup_status)


# NB-07 — Observe the target, not necessarily the whole state
Companion to Volume IV §13, Volume III §§6–7 and Volume VI §5.
Prerequisites: NB-04 and NB-06. We reuse the retained six-mode System E
diffusion experiment: three observation times, a fixed target horizon, and
the saved matched/mismatch results. No new observations or noise are sampled.

A useful sensor design depends on the target. We compare permanent locations
for regional, interface and local targets, then inspect what the declared
dynamics and omitted modes can hide.

In [ ]:
NOTEBOOK_ID='NB-07'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

spec=inputs['diagram'];targets=inputs['vertex_targets']
ref=dict(retained_modes=inputs['retained_modes'],times=inputs['observation_times'],horizon=inputs['target_horizon'],beta=inputs['beta'],rows=inputs['retained_monte_carlo_summaries'])
print('Local original network inspection:',spec['lines'][:2])
print('Identity columns below are deterministic operator queries, not observations or simulated noise.')
def construct(label,times,k):
    graph=execute(label+':graph','graph_operators',inputs['graph_requests'][label])['result']['laplacian_bundle']
    nodes=graph['node_ids'];n=len(nodes)
    assert graph['definition_id']=='construct.graph.combinatorial_laplacian.v1'
    assert graph['context']['operator_unit']=='1/model_time'
    context={key:graph['context'][key] for key in ('model_context_id','topology_id','valid_support_id')}
    context.update(inner_product='Euclidean',nullspace='constant_one_connected_component',operator_unit='1/model_time',interpretation='declared_diffusion_rate')
    op=dict(node_ids=nodes,values=graph['values'],definition_id=graph['definition_id'],context=context)
    def filter_request(values,cases,variant,tau=None):
        return dict(inputs=dict(laplacian_operator=op,graph_signal=dict(node_ids=nodes,case_ids=cases,values=values,unit='declared_operator_query',context=context),filter_response=dict(variant=variant,tau=tau,tau_unit='model_time' if tau is not None else None,spectral_coordinate='graph_laplacian_eigenvalue',retained_modes='all',nullspace_policy='preserve')),parameters={},context=dict(input_origin='NB-07 original declared diffusion model; explicit source-result selection and model-time interpretation; no empirical rate claim'))
    coordinate_results=[];parents=[]
    cases=['query:'+node for node in nodes]
    for index,time in enumerate(times):
        heatname=label+':heat:'+str(index);coordname=label+':coordinates:'+str(index)
        heat=execute(heatname,'graph_filter',filter_request(np.eye(n).tolist(),cases,'heat',time),[label+':graph'])['result']['graph_signal']
        assert heat['case_ids']==cases and heat['node_ids']==nodes
        coords=execute(coordname,'graph_filter',filter_request(heat['values'],cases,'coordinates'),[label+':graph',heatname])['result']['modal_coefficients']
        coordinate_results.append(coords);parents.append(coordname)
    basis=coordinate_results[0]['basis']
    assert all(c['basis']['basis_id']==basis['basis_id'] and c['case_ids']==cases for c in coordinate_results)
    # Explicitly select the original first k coordinates, preserving returned basis identity.
    values=np.stack([np.asarray(c['values'])[:,:k] for c in coordinate_results],axis=1).tolist()
    block=dict(candidate_ids=nodes,mode_ids=basis['mode_ids'][:k],basis_context=basis['basis_id'],values=values,observation_times=times,time_unit='model_time',model='declared_diffusion_blocks')
    return block,basis,parents,filter_request
block,basis,block_parents,filter_request=construct('system_e',ref['times'],6)
names=list(targets)
heat=execute('targets:heat','graph_filter',filter_request([targets[name] for name in names],names,'heat',ref['horizon']),['system_e:graph'])['result']['graph_signal']
coordinates=execute('targets:coordinates','graph_filter',filter_request(heat['values'],names,'coordinates'),['system_e:graph','targets:heat'])['result']['modal_coefficients']
assert coordinates['basis']['basis_id']==basis['basis_id'] and coordinates['case_ids']==names
budget=dict(count=6,beta=ref['beta'],tie_policy='input_index',optimality='greedy_only')
full_request=dict(inputs=dict(measurement_blocks=block,design_budget=budget),parameters={},context=dict(input_origin='NB-07 original full-state regularized trace objective, distinct from scalar-target design'))
full=execute('full_state','full_state_design',full_request,block_parents)['result']['sensor_design']
assert full['definition_id']=='optimize.observation.full_state_design.v1'
assert not full['scientific_acceptance'] and not full['statistical_model_certified']
rows=[];endpoints=[];target_requests={}
for index,name in enumerate(names):
    fixed=[dict(label='full_state_'+str(count),candidate_ids=full['selected_ids'][:count]) for count in range(1,7)]
    if name in ('interface','local'):
        fixed.append(dict(label='endpoint',candidate_ids=['7','12'] if name=='interface' else ['14','15']))
    target=dict(mode_ids=block['mode_ids'],basis_context=block['basis_context'],values=coordinates['values'][index][:6],unit='declared_target')
    request=dict(inputs=dict(measurement_blocks=block,target_coordinates=target,design_budget=budget),parameters=dict(diagnostic_profile='retained_design_evaluation',fixed_candidate_sets=fixed),context=dict(input_origin='NB-07 original scalar target, explicit first-six-mode selection; fixed sets compared at matched location budgets'))
    target_requests[name]=request
    output=execute('target:'+name,'target_design',request,block_parents+['targets:coordinates','full_state'])
    diagnostics=output['diagnostics']['retained_design_evaluation']
    for method,items in [('target',diagnostics['greedy_prefixes'][1:]),('full_state',diagnostics['fixed_sets'][:6])]:
        for count,fit in enumerate(items,1):
            rows.append(dict(target=name,budget=count,method=method,locations=fit['selected_indices'],risk=fit['normalized_risk_squared'],span_error=fit['span_error'],rank=fit['rank']))
    if name in ('interface','local'):
        fit=diagnostics['fixed_sets'][-1]
        endpoints.append(dict(target=name,budget=2,locations=fit['selected_indices'],risk=fit['normalized_risk_squared'],span_error=fit['span_error'],rank=fit['rank']))
triangle,tbasis,tparents,_=construct('model_b',inputs['model_b_times'],3)
repeated_request=dict(inputs=dict(measurement_blocks=triangle,target_coordinates=dict(mode_ids=triangle['mode_ids'],basis_context=triangle['basis_context'],values=triangle['values'][0][0],unit='initial_location_target'),design_budget=dict(count=1,beta=.1,tie_policy='input_index',optimality='greedy_only')),parameters=dict(diagnostic_profile='retained_design_evaluation',fixed_candidate_sets=[dict(label='original_location_zero',candidate_ids=['0'])]),context=dict(input_origin='Existing Model B repeated-eigenvalue operator control; all three original modes and times; no observations generated'))
repeated=execute('repeated_control','target_design',repeated_request,tparents)['diagnostics']['retained_design_evaluation']['fixed_sets'][0]
obstruction=dict(spectrum=tbasis['eigenvalues'],rank=repeated['rank'],dimension=3,target_span_error=repeated['span_error'])
print('Retained modes:',ref['retained_modes'],'observation times:',ref['times'],'target horizon:',ref['horizon'],'beta:',ref['beta'])
print('All 36 analytic designs are recomputed privately; source parity is checked in the retained validation record.')

## State coordinates, measurement blocks and target row space
Under this declared model, x(t)=Uₖ exp(−tΛₖ)a. For location i, Bᵢ stacks the
three observation rows. The horizon target is gᵀa, where
g=exp(−0.2Λₖ)Uₖᵀc. The six-mode set includes the constant mode.

For a stack B, choose w to minimize ‖Bᵀw−g‖²+β‖w‖². This equals mean squared
error only under the declared unit-covariance coefficients and independent
observation noise with variance β. A zero row-space residual means the noiseless
target is identifiable; it does not guarantee small noisy risk or full-state
observability. We verify both primal and dual linear solves.

The graph is read from the saved rounded specification. Eigenvectors are
recomputed, so we check invariant risks and locations rather than signs.

In [ ]:
for name in targets:
    one=next(r for r in rows if (r['target'],r['budget'],r['method'])==(name,1,'target'))
    print(name,{k:one[k] for k in ('locations','rank','span_error','risk')})
print('Existing Model B repeated-eigenvalue control:',obstruction)
assert obstruction['rank']==2 and obstruction['target_span_error']<1e-12
# Same existing test case: three samples at one location cannot resolve all
# three modes when the two nonconstant eigenvalues coincide.
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,name in zip(axes,targets):
    for method,color in [('target','#88C0D0'),('full_state','#B48EAD')]:
        subset=[r for r in rows if r['target']==name and r['method']==method]
        ax.plot([r['budget'] for r in subset],[r['risk'] for r in subset],'o-',label=method,color=color)
    ax.set(xlabel='Permanent locations',ylabel='Risk / matched target variance',title=name.replace('_',' '),yscale='log');ax.legend(fontsize=8)
fig.suptitle('Analytic risk · identical times and noise at each budget');fig.tight_layout();plt.show()

## Matched budgets and direct endpoint locations
The target design greedily reduces that target's risk. The full-state design
greedily reduces the trace of the regularized state inverse. Neither procedure
is certified as the global optimum. Each installed location supplies the same
three measurements; budget counts locations, not scalar readings.

For each spread, a direct-endpoint **location** baseline installs sensors at
both endpoints using the same times, noise and horizon. The linear forecast
still uses the declared dynamics; it is not direct observation of a future
spread. No new sensor noise is generated for this analytic comparison.

In [ ]:
for endpoint in endpoints:
    comparator=next(r for r in rows if (r['target'],r['budget'],r['method'])==(endpoint['target'],2,'target'))
    print(endpoint['target'],'endpoint locations',endpoint['locations'],'risk',round(endpoint['risk'],6),'target-design risk',round(comparator['risk'],6))
name='interface';selected=next(r['locations'] for r in rows if (r['target'],r['budget'],r['method'])==(name,2,'target'))
xy=np.array([[b['x'],b['y']] for b in spec['buses']]);fig,ax=plt.subplots(figsize=(9,5))
for line in spec['lines']:
    i,j=line['from'],line['to'];ax.plot(xy[[i,j],0],xy[[i,j],1],color='#4C566A',lw=.8,zorder=1)
ax.scatter(xy[:,0],xy[:,1],color='#D8DEE9',s=25,label='Bus')
ax.scatter(xy[selected,0],xy[selected,1],color='#88C0D0',s=130,facecolors='none',linewidths=2,label='Target design: 2 locations')
ends=[7,12];ax.scatter(xy[ends,0],xy[ends,1],color='#D08770',s=90,marker='x',label='Spread endpoints')
for b,(x,y) in zip(spec['buses'],xy):ax.annotate(str(b['id']),(x,y),xytext=(3,3),textcoords='offset points',fontsize=8)
ax.set(title='Existing System E layout · zero-based bus IDs',xlabel='Illustrative x',ylabel='Illustrative y');ax.legend(fontsize=8);fig.tight_layout();plt.show()

## Repeated observations do not remove every obstruction
The Model B control has spectrum (0,3,3). Repeated times scale both modes in
the repeated eigenspace identically, so one permanent sensor cannot distinguish
both directions. Its block rank is two, yet its own initial-location target
lies in the row space. Target identifiability can therefore hold while full
state recovery fails. These are exact operator calculations, not new datasets.

In System E, even rank-six endpoint observations can give substantial local
target risk because the inverse is poorly conditioned. Rank is not accuracy.

## Replayed mismatch results: models matter more than the rank label
The retained experiment used 4,000 held-out coefficient/noise realizations per
case, shared across methods. We load its aggregate MSEs and Monte Carlo standard
errors. The individual realizations were not retained in this bundle, so these
empirical MSEs are **replayed**, not independently recomputed here. Analytic
risks and selected locations above are recomputed.

Stresses retain the original outage, correlated-noise and omitted-mode cases.
Every denominator is the matched target variance, including stressed results.
The local target's small matched variance can make normalized stress errors
especially large. Correlated noise need not worsen every target.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,name in zip(axes,targets):
    for stress,color in [('matched','#88C0D0'),('outage','#D08770'),('correlated_noise','#A3BE8C'),('unmodeled_modes','#B48EAD')]:
        subset=[r for r in ref['rows'] if r['target']==name and r['method']=='target' and r['stress']==stress]
        ax.errorbar([r['budget'] for r in subset],[r['normalized_mse'] for r in subset],yerr=[1.96*r['mse_se'] for r in subset],marker='o',label=stress,color=color,markersize=3)
    ax.set(xlabel='Permanent locations',ylabel='MSE / matched target variance',title=name.replace('_',' '),yscale='log');ax.legend(fontsize=7)
fig.suptitle('Retained Monte Carlo results · ±1.96 standard errors, not market uncertainty');fig.tight_layout();plt.show()
local=[r for r in ref['rows'] if r['target']=='local' and r['method']=='target' and r['budget']==2]
for r in local:print(r['stress'],round(r['normalized_mse'],6),'normalized MSE; nominal risk',round(r['model_risk'],6))

## Matched Monte Carlo results versus the analytic prediction
Agreement is a check of the declared synthetic model. It does not demonstrate
that electricity prices follow graph diffusion. A real application requires
fitted transition and observation models, valid time cutoffs, sensor economics
and separate out-of-sample target/decision evaluation.

In [ ]:
matched=[r for r in ref['rows'] if r['stress']=='matched'];fig,ax=plt.subplots(figsize=(7,5))
for method,color in [('target','#88C0D0'),('full_state','#B48EAD')]:
    subset=[r for r in matched if r['method']==method]
    ax.scatter([r['model_risk'] for r in subset],[r['normalized_mse'] for r in subset],label=method,color=color)
limits=[min(r['model_risk'] for r in matched),max(r['model_risk'] for r in matched)]
ax.plot(limits,limits,'--',color='#A3BE8C');ax.set(xscale='log',yscale='log',xlabel='Analytic normalized risk',ylabel='Retained normalized MSE',title='Matched-model agreement across targets and budgets');ax.legend();fig.tight_layout();plt.show()

## Exercise
Compare the local target at budget two under matched and omitted-mode cases.
Explain why adding sensors selected under the wrong dynamics need not repair
the omitted-state problem. Then distinguish zero target row-space residual,
full rank, low model risk, and demonstrated predictive value.

In [ ]:
bad=copy.deepcopy(full_request);bad['inputs']['measurement_blocks']['mode_ids']=list(reversed(block['mode_ids']));bad['inputs']['measurement_blocks']['values'][0][0].pop();reject('full_state_design',bad)
bad=copy.deepcopy(target_requests['interface']);bad['parameters']['fixed_candidate_sets'][0]['candidate_ids']=['unknown-location'];reject('target_design',bad)
checked=dict(saved_design_locations=True,analytic_risk_parity=True,primal_dual_agreement=True,repeated_eigenvalue_obstruction=True,matched_endpoint_baseline=True,stress_results_replayed_not_regenerated=True,new_synthetic_data=False,scientific_acceptance=False)
(session.output/'consumer-summary.json').write_text(json.dumps(dict(analytic_designs=rows,endpoint_baselines=endpoints,repeated_eigenvalue=obstruction),indent=2,allow_nan=False)+'\n')
summary=session.finish(checked)
print(summary)